# X03 IOI 电路


## 目标

把电路分析从 toy 任务推进到自然语言里的具体行为现象，是“从论文图到真实行为任务”的关键过渡。


## 为什么现在学这篇

做完 Anthropic 主线后，这篇能逼你处理更脏的行为定义、更多的头和更复杂的证据链。


## Notebook 与交付

- 原文: https://arxiv.org/abs/2211.00593
- Notebook：`notebooks/extensions/zh/x03_ioi_circuit.ipynb`
- 先修: X02, M06
- 在 Colab 里复现一个教学版 IOI 证据链，并写 1 份 evidence-chain 速记，指出这类行为任务比 M06 toy trace 多了哪些不确定性。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

roles = np.array([
    [1.0, 0.0],  # subject = Alice
    [0.0, 1.0],  # indirect object = Bob
])
name_labels = ["Alice", "Bob"]
duplicate_head = np.array([[0.0, 0.2], [0.1, 0.0]])
name_mover = np.array([[0.2, 0.0], [0.0, 1.3]])
subject_suppressor = np.array([[1.0, 0.0], [0.0, 0.1]])

baseline_scores = np.array([0.5, 0.5])
full_scores = baseline_scores + roles[1] @ name_mover - roles[0] @ subject_suppressor + roles[1] @ duplicate_head
ablations = {
    "full": full_scores,
    "without name mover": baseline_scores - roles[0] @ subject_suppressor + roles[1] @ duplicate_head,
    "without subject suppressor": baseline_scores + roles[1] @ name_mover + roles[1] @ duplicate_head,
    "without duplicate head": baseline_scores + roles[1] @ name_mover - roles[0] @ subject_suppressor,
}

fig, ax = plt.subplots(figsize=(9, 4))
positions = np.arange(len(ablations))
width = 0.35
for idx, name in enumerate(name_labels):
    ax.bar(
        positions + (idx - 0.5) * width,
        [scores[idx] for scores in ablations.values()],
        width=width,
        label=name,
    )
ax.set_xticks(positions, list(ablations.keys()), rotation=15)
ax.set_ylabel("final logit score")
ax.set_title("Teaching-scale IOI evidence chain")
ax.legend()
plt.tight_layout()

for label, scores in ablations.items():
    winner = name_labels[int(np.argmax(scores))]
    print(label, "->", winner, np.round(scores, 3))


## 小结

到了行为层面的电路分析，不能只靠一张漂亮的路径图，还要靠消融来补证据。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个教学版 IOI 证据链，并写 1 份 evidence-chain 速记，指出这类行为任务比 M06 toy trace 多了哪些不确定性。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 为什么 IOI 这类行为任务比 toy trace 更难建立证据闭环？
- 在你的教学版证据链里，哪一条边最像“必要但不充分”的证据？
- 如果一个头在图里很亮，你还需要什么额外证据才敢说它负责这个行为？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
